# Semaine 2 — Jour 3 : Structured Outputs

Notebook étudiant généré à partir du Markdown source.

## Objectifs

- Concevoir un schéma de sortie.
- Parser et valider une sortie JSON.
- Mapper une donnée validée vers un objet métier.
- Tester les cas valides et invalides.

## Modèle mental

```text
User request → Model output → JSON parsing → Schema validation → Domain object → Business decision
```

## Extrait du chapitre


# Chapitre — Structured Outputs

## 1. Le problème des sorties non structurées

Un modèle de langage produit naturellement du texte.

Pour une démonstration, cela suffit :

```text
Le ticket semble urgent et concerne probablement un problème de facturation.
```

Pour un backend, cette phrase est insuffisante.

Une application doit souvent prendre des décisions concrètes :

- router un ticket vers une équipe ;
- déclencher une alerte ;
- créer une tâche ;
- renseigner une base de données ;
- calculer un SLA ;
- afficher une interface stable ;
- exécuter un workflow.

Dans ces cas, le texte libre devient un risque.

Le backend doit deviner :

- où se trouve la priorité ;
- quelle équipe est concernée ;
- si l’action est obligatoire ;
- quelle partie est un résumé ;
- si la valeur de confiance est exploitable ;
- si la réponse contient un format inattendu.

Un agent IA professionnel doit donc produire des données structurées, pas seulement du texte.

## 2. Définition

Un **Structured Output** est une réponse de modèle contrainte par un schéma.

Le schéma définit :

- la forme attendue ;
- les champs obligatoires ;
- les types ;
- les valeurs autorisées ;
- les objets imbriqués ;
- les contraintes de validation ;
- ce qui est interdit.

Exemple de sortie structurée :

```json
{
  "category": "billing",
  "priority": "high",
  "sentiment": "frustrated",
  "summary": "Le client signale une double facturation.",
  "action_required": true,
  "next_action": {
    "owner_team": "billing_ops",
    "rationale": "Le ticket mentionne un paiement prélevé deux fois."
  },
  "confidence": 0.91
}
```

Cette sortie est immédiatement exploitable par un programme.

## 3. Différence avec le Function Calling

Le jour précédent a introduit le Function Calling.

Le modèle produit alors une intention d’appel :

```json
{
  "name": "get_order_status",
  "arguments": {
    "order_id": "ORD-1001"
  }
}
```

L’application exécute ensuite l’outil.

Les Structured Outputs répondent à un besoin différent : contrôler la forme de la réponse finale ou intermédiaire.

Comparaison :

| Mécanisme | Question principale | Produit par le modèle | Utilisé pour |
|---|---|---|---|
| Texte libre | Que répondre ? | Phrase ou paragraphe | Chat simple |
| JSON libre | Peux-tu répondre en JSON ? | JSON non garanti | Prototype |
| JSON mode | Peux-tu produire du JSON valide ? | JSON syntaxiquement valide | Réponses simples |
| Function Calling | Quel outil appeler ? | Nom d’outil + arguments | Action externe |
| Structured Outputs | Quelle donnée conforme au schéma produire ? | Objet conforme au contrat | Intégration backend |

La distinction clé est la suivante :

> Le Function Calling structure une action. Les Structured Outputs structurent une donnée.

## 4. Pourquoi le JSON libre ne suffit pas

Demander à un modèle :

```text
Réponds uniquement en JSON.
```

ne suffit pas pour un système de production.

Le modèle peut produire :

```json
{
  "categorie": "facturation",
  "priority": "urgent",
  "confidence": "very high"
}
```

Ce JSON est syntaxiquement valide, mais il casse le contrat attendu :

- `categorie` est en français alors que le backend attend `category` ;
- `urgent` n’est peut-être pas une priorité autorisée ;
- `confidence` est une chaîne au lieu d’un nombre ;
- des champs obligatoires peuvent manquer.

Le problème n’est donc pas seulement la syntaxe JSON. Le problème est la conformité au contrat métier.

## 5. Le schéma comme contrat

Un schéma de sortie doit être traité comme une API interne.

Il doit répondre à ces questions :

- quels champs sont obligatoires ?
- quels types sont acceptés ?
- quelles valeurs sont autorisées ?
- quelles contraintes numériques existent ?
- les champs supplémentaires sont-ils interdits ?
- comment versionner le contrat ?
- qui consomme cette donnée ?

Exemple de contrat métier :

```json
{
  "type": "object",
  "required": [
    "category",
    "priority",
    "sentiment",
    "summary",
    "action_required",
    "next_action",
    "confidence"
  ],
  "additionalProperties": false,
  "properties": {
    "category": {
      "type": "string",
      "enum": ["billing", "technical", "account", "shipping", "other"]
    },
    "priority": {
      "type": "string",
      "enum": ["low", "medium", "high", "critical"]
    },
    "sentiment": {
      "type": "string",
      "enum": ["neutral", "frustrated", "angry", "satisfied"]
    },
    "summary": {
      "type": "string"
    },
    "action_required": {
      "type": "boolean"
    },
    "next_action": {
      "type": "object",
      "required": ["owner_team", "rationale"],
      "additionalProperties": false,
      "properties": {
        "owner_team": {
          "type": "string",
          "enum": ["support_l1", "billing_ops", "technical_ops", "account_ops"]
        },
        "rationale": {
          "type": "string"
        }
      }
    },
    "confidence": {
      "type": "number",
      "minimum": 0,
      "maximum": 1
    }
  }
}
```

## 6. Architecture d’un pipeline Structured Output

Un pipeline robuste sépare plusieurs responsabilités :

```text
Prompt builder
→ Model adapter
→ JSON parser
→ Schema validator
→ Domain mapper
→ Business rule engine
```

Chaque composant a un rôle précis.

### Prompt builder

Il prépare la demande.

Il doit expliquer :

- la tâche ;
- les contraintes métier ;
- le schéma attendu ;
- les valeurs autorisées ;
- les exemples utiles ;
- les règles de refus ou d’incertitude.

### Model adapter

Il encapsule l’appel au modèle.

Dans un lab local, ce composant peut être remplacé par un faux modèle déterministe.

En production, il isole :

- le fournisseur LLM ;
- le modèle utilisé ;
- les paramètres ;
- les erreurs réseau ;
- les mécanismes de retry.

### JSON parser

Il convertit la chaîne produite en objet Python.

Il doit échouer proprement si la sortie n’est pas du JSON valide.

### Schema validator

Il vérifie la conformité au contrat.

Il doit détecter :

- champs manquants ;
- types incorrects ;
- valeurs non autorisées ;
- nombres hors bornes ;
- propriétés inattendues ;
- objets imbriqués invalides.

### Domain mapper

Il convertit le dictionnaire validé en objet métier.

Par exemple :

```python
TriageDecision(
    category="billing",
    priority="high",
    sentiment="frustrated",
    summary="Le client signale une double facturation.",
    action_required=True,
    owner_team="billing_ops",
    confidence=0.91,
)
```

### Business rule engine

Il applique les règles internes.

Exemple :

```text
Si priority = critical et confidence >= 0.85
→ créer une alerte immédiate
```

Le modèle ne doit pas être le seul responsable des règles critiques.



In [ ]:
from pathlib import Path
import sys

lab_path = Path.cwd() / 'book' / 'week02' / 'day03' / 'labs'
if lab_path.exists():
    sys.path.append(str(lab_path))
else:
    # When running the notebook from notebooks/week02, go back to repository root.
    sys.path.append(str(Path.cwd().parents[1] / 'book' / 'week02' / 'day03' / 'labs'))

from structured_output_agent import triage_ticket, build_user_reply

In [ ]:
ticket = "Je suis très énervé, vous m’avez facturé deux fois ce mois-ci."
decision = triage_ticket(ticket)
decision

In [ ]:
print(build_user_reply(decision))

## Exercices


# Exercices — Structured Outputs

## Exercice 1 — Identifier la bonne stratégie

Pour chaque besoin, indique le mécanisme le plus adapté :

- texte libre ;
- JSON mode ;
- function calling ;
- Structured Outputs.

### Cas A

L’utilisateur demande une explication pédagogique sur le fonctionnement d’un transformeur.

### Cas B

L’application doit appeler une fonction `get_invoice(invoice_id)`.

### Cas C

Le backend doit recevoir une décision de routage contenant `category`, `priority` et `owner_team`.

### Cas D

Un prototype doit simplement produire un JSON valide pour une démonstration interne non critique.

## Exercice 2 — Concevoir un schéma

Conçois un schéma JSON pour classifier une demande utilisateur avec les champs suivants :

- `intent` : parmi `order_status`, `refund`, `technical_issue`, `other` ;
- `requires_tool` : booléen ;
- `missing_fields` : liste de chaînes ;
- `confidence` : nombre entre 0 et 1.

Contraintes :

- tous les champs sont obligatoires ;
- aucune propriété supplémentaire n’est acceptée ;
- les valeurs de `intent` sont strictement limitées.

## Exercice 3 — Détecter les erreurs

Le schéma attendu impose :

```json
{
  "category": "billing | technical | account | shipping | other",
  "priority": "low | medium | high | critical",
  "confidence": "number between 0 and 1"
}
```

La sortie suivante est produite :

```json
{
  "category": "finance",
  "priority": "urgent",
  "confidence": "0.9",
  "explanation": "Client très mécontent"
}
```

Liste toutes les erreurs de contrat.

## Exercice 4 — Mapper vers un objet métier

À partir de la sortie suivante :

```json
{
  "category": "technical",
  "priority": "high",
  "sentiment": "frustrated",
  "summary": "Le client ne peut plus se connecter.",
  "action_required": true,
  "next_action": {
    "owner_team": "technical_ops",
    "rationale": "Le problème concerne l'accès au compte."
  },
  "confidence": 0.88
}
```

Décris l’objet métier Python que tu créerais.

Tu n’as pas besoin d’écrire tout le code, mais tu dois préciser :

- le nom de la classe ;
- les champs ;
- les types attendus ;
- ce qui doit avoir été validé avant instanciation.

## Exercice 5 — Stratégie de récupération

Une sortie structurée échoue parce que le modèle a oublié le champ `confidence`.

Propose une stratégie de récupération robuste.

Ta réponse doit inclure :

- ce que le système journalise ;
- ce qu’il redemande au modèle ;
- le fallback si la deuxième tentative échoue ;
- pourquoi il ne faut pas inventer une confiance par défaut sans le signaler.


## Challenge


# Challenge — Agent de triage support avec sortie structurée

## Contexte

Tu construis un assistant IA mono-agent pour une équipe support.

L’agent reçoit un ticket utilisateur et doit produire une décision structurée, validable et exploitable par un backend.

## Objectif

Implémenter ou compléter un pipeline qui transforme un ticket en `TriageDecision`.

Le pipeline doit :

1. produire une sortie JSON ;
2. parser cette sortie ;
3. valider le contrat ;
4. convertir la sortie validée en objet Python ;
5. refuser explicitement les sorties invalides.

## Schéma attendu

La sortie doit respecter le contrat suivant :

```json
{
  "category": "billing | technical | account | shipping | other",
  "priority": "low | medium | high | critical",
  "sentiment": "neutral | frustrated | angry | satisfied",
  "summary": "string",
  "action_required": "boolean",
  "next_action": {
    "owner_team": "support_l1 | billing_ops | technical_ops | account_ops",
    "rationale": "string"
  },
  "confidence": "number between 0 and 1"
}
```

Tous les champs sont obligatoires.

Aucune propriété supplémentaire n’est autorisée.

## Contraintes

- Ne pas utiliser de dépendance externe.
- Ne pas faire confiance au JSON sans validation.
- Ne pas corriger silencieusement les valeurs invalides.
- Écrire au moins trois tests :
  - un cas valide ;
  - un cas avec enum invalide ;
  - un cas avec champ manquant.
- Le code doit rester lisible pour un AI Backend Engineer junior.

## Scénarios à couvrir

### Scénario 1 — Double facturation

```text
Je suis très énervé, vous m’avez facturé deux fois ce mois-ci.
```

Attendu :

- catégorie : `billing` ;
- priorité : `high` ou `critical` ;
- sentiment : `angry` ou `frustrated` ;
- équipe : `billing_ops`.

### Scénario 2 — Problème de connexion

```text
Depuis ce matin je ne peux plus me connecter à mon compte.
```

Attendu :

- catégorie : `technical` ou `account` ;
- priorité : au moins `medium` ;
- équipe : `technical_ops` ou `account_ops`.

### Scénario 3 — Question simple

```text
Bonjour, je voudrais savoir où trouver mes anciennes factures.
```

Attendu :

- catégorie : `billing` ;
- priorité : `low` ou `medium` ;
- action requise : `true` ;
- équipe : `billing_ops` ou `support_l1`.

## Extension facultative

Ajoute une fonction `build_user_reply(decision)` qui génère une réponse utilisateur à partir de la décision validée.

La réponse utilisateur ne doit pas être mélangée avec le JSON machine.
